# Downloading and loading mental health dataset from kaggle
* download the python kaggle library
* pip install kaggle
* sign into kaggle and create a token

In [1]:
! pip install kaggle

In [2]:
import kaggle

In [3]:
# changing the permission of kaggle file so that only the user on this device can read and write the file.
! chmod 755 /home/marius/.kaggle/kaggle.json

In [9]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

In [10]:
# Downloading the dataset into the data folder
api.dataset_download_files('divaniazzahra/mental-health-dataset', path='./data', unzip=True)

Dataset URL: https://www.kaggle.com/datasets/divaniazzahra/mental-health-dataset


In [ ]:
def download_kaggle_dataset(dataset, download_path='./data'):
    """
    Automates the download of a dataset from Kaggle.
    
    Args:
        dataset (str): The Kaggle dataset identifier (e.g., 'user/dataset').
        download_path (str): The directory where the file should be downloaded.
    """
    api.dataset_download_files(dataset, path=download_path, unzip=True)
    print(f"Dataset '{dataset}' has been downloaded and unzipped to '{download_path}'.")

# Example usage:
# download_kaggle_dataset('divaniazzahra/mental-health-dataset')

In [15]:
import pyspark
from pyspark.sql import SparkSession

In [14]:
key_path = "/home/marius/mental_health/DE_PROJECT1/key/teraform-mar-0fb97fcd6586.json"

In [16]:
from pyspark.sql import SparkSession

# Path to your key (based on your traceback)
key_path = "/home/marius/mental_health/DE_PROJECT1/key/teraform-mar-0fb97fcd6586.json"

spark = SparkSession.builder \
    .appName("MentalHealthDataWrite") \
    .config("spark.jars.packages", "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.5") \
    .getOrCreate()

# Set Hadoop configurations to use the Service Account JSON key
conf = spark._jsc.hadoopConfiguration()
conf.set("google.cloud.auth.service.account.enable", "true")
conf.set("google.cloud.auth.service.account.json.keyfile", key_path)
conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")


26/04/10 17:25:51 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [9]:
# checking the version of spark
spark.version

'3.5.0'

In [10]:
# defining the schema for the mental health dataset
from pyspark.sql import types

schema = types.StructType([
    types.StructField("Timestamp", types.TimestampType(), True),
    types.StructField("Gender", types.StringType(), True),
    types.StructField("Country", types.StringType(), True),
    types.StructField("Occupation", types.StringType(), True),
    types.StructField("self_employed", types.StringType(), True),
    types.StructField("family_history", types.StringType(), True),
    types.StructField("treatment", types.StringType(), True), # Changed to StringType since CSV has 'Yes'/'No'
    types.StructField("Days_Indoors", types.StringType(), True), # Changed to StringType for ranges like '1-14 days'
    types.StructField("Growing_Stress", types.StringType(), True),
    types.StructField("Changes_Habits", types.StringType(), True),
    types.StructField("Mental_Health_History", types.StringType(), True),
    types.StructField("Mood_Swings", types.StringType(), True),
    types.StructField("Coping_Struggles", types.StringType(), True),
    types.StructField("Work_Interest", types.StringType(), True),
    types.StructField("Social_Weakness", types.StringType(), True),
    types.StructField("mental_health_interview", types.StringType(), True),
    types.StructField("care_options", types.StringType(), True),
])

In [11]:
# running spark command to read the .csv in the current directory
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('data/Mental Health Dataset.csv')

In [12]:
df.show(5)

+-------------------+------+-------------+----------+-------------+--------------+---------+------------+--------------+--------------+---------------------+-----------+----------------+-------------+---------------+-----------------------+------------+
|          Timestamp|Gender|      Country|Occupation|self_employed|family_history|treatment|Days_Indoors|Growing_Stress|Changes_Habits|Mental_Health_History|Mood_Swings|Coping_Struggles|Work_Interest|Social_Weakness|mental_health_interview|care_options|
+-------------------+------+-------------+----------+-------------+--------------+---------+------------+--------------+--------------+---------------------+-----------+----------------+-------------+---------------+-----------------------+------------+
|2014-08-27 11:29:31|Female|United States| Corporate|         NULL|            No|      Yes|   1-14 days|           Yes|            No|                  Yes|     Medium|              No|           No|            Yes|                     N

In [17]:
# Write DataFrame to GCS bucket in Parquet format
# Note: Replace 'mar_mental_health_bucket' with your actual bucket name if different
df.write.mode("overwrite").parquet("gs://mar_mental_health_bucket/mental_health_data/")